# 集合覆盖 scp41（最小成本）

**问题**：实例来自 OR-Library 的 scp41：m=200 个行元素，n=1000 个列集合。列 j 的成本为 c_j，覆盖的行集合为 S_j（a_ij=1 表示列 j 覆盖行 i）。目标是选择一组列，使每一行至少被一个选中列覆盖，同时总成本最小。

**数学模型**

$$\min \sum_{j=1}^{n} c_j x_j$$

$$\text{s.t.}\quad \sum_{j: i \in S_j} x_j \ge 1,\quad i=1,\dots,m$$

$$x_j\in\{0,1\},\quad j=1,\dots,n$$

数据文件：\`/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt\`。文献最优值 429（本套件用直接 MIP 自证）。

## 方法：LBBD（逻辑 Benders 分解）

**主问题**：选列 y_j∈{0,1}，目标 min Σ c_j y_j，初始不加入覆盖约束。

**子问题**：给定主问题解 y，检查每个行 r 是否至少被一个选中列覆盖；返回所有未覆盖行集合 U。

**逻辑割**：对每个未覆盖行 r 加入

$$\sum_{j: r\in S_j} y_j \ge 1$$

这条割正是覆盖问题中“第 r 行必须被覆盖”的逻辑 Benders cut。它由子问题可行性检查直接导出，不需要对偶信息。

**原理要点**

1. 主问题用 MathOpt + HIGHS 解整数规划；子问题是纯逻辑可行性检查。
2. 初始主问题无覆盖约束，解得空集（成本 0）；子问题返回全部 200 个未覆盖行。
3. 把这些行对应的逻辑割加入主问题后，主问题等价于原 SCP，解得最优覆盖。
4. 迭代至主问题解覆盖全部行（无未覆盖行），此时主问题最优解即原问题最优解。
5. 停机：无未覆盖行、迭代上限 50、总墙钟 110s。

In [1]:
import platform, time, datetime, math, ortools
from ortools.math_opt.python import mathopt

print("python", platform.python_version(), "| ortools", ortools.__version__)

DATA = "/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt"
toks = open(DATA).read().split()
m, n = map(int, toks[:2])
costs = list(map(int, toks[2:2+n]))
idx = 2 + n
rows = []
for _ in range(m):
    k = int(toks[idx]); idx += 1
    rows.append([int(t)-1 for t in toks[idx:idx+k]]); idx += k
assert idx == len(toks)
colrows = [[] for _ in range(n)]
for i, row in enumerate(rows):
    for j in row:
        colrows[j].append(i)
print("m,n =", m, n, "| rows parsed =", len(rows), "| tokens consumed =", idx)


python 3.10.20 | ortools 9.15.6755
m,n = 200 1000 | rows parsed = 200 | tokens consumed = 5211


In [2]:
cut_rows = set()
max_iter = 50
t0 = time.perf_counter()
for it in range(max_iter):
    model = mathopt.Model(name="lbbd_master")
    y = [model.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"y{j}") for j in range(n)]
    model.minimize_linear_objective(sum(costs[j]*y[j] for j in range(n)))
    for r in cut_rows:
        model.add_linear_constraint(sum(y[j] for j in rows[r]) >= 1.0, name=f"cut{r}")
    res = mathopt.solve(model, mathopt.SolverType.HIGHS, params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=120), enable_output=False))
    yv = res.variable_values(y)
    sel = [j for j in range(n) if yv[j] > 0.5]
    uncovered = []
    for r, row in enumerate(rows):
        if not any(yv[j] > 0.5 for j in row):
            uncovered.append(r)
    print(f"iter {it+1}: master_obj={res.objective_value()}, selected={len(sel)}, uncovered={len(uncovered)}, cuts={len(cut_rows)}")
    if not uncovered:
        print("all rows covered -> LBBD convergence")
        break
    for r in uncovered:
        cut_rows.add(r)
    if time.perf_counter()-t0 > 110:
        print("time limit reached")
        break
wall = time.perf_counter()-t0
print("LBBD wall:", round(wall, 3), "| iterations:", it+1, "| total logic cuts:", len(cut_rows), "| final obj:", res.objective_value())
print("termination:", res.termination.reason)
print("selected_columns:", sorted(sel))


iter 1: master_obj=0.0, selected=0, uncovered=200, cuts=0


iter 2: master_obj=429.0, selected=66, uncovered=0, cuts=200
all rows covered -> LBBD convergence
LBBD wall: 0.203 | iterations: 2 | total logic cuts: 200 | final obj: 429.0
termination: TerminationReason.OPTIMAL
selected_columns: [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 24, 25, 27, 28, 42, 43, 45, 46, 47, 48, 49, 51, 53, 57, 58, 61, 62, 65, 68, 69, 70, 74, 76, 77, 80, 84, 85, 88, 90, 93, 102, 106, 115, 119, 120, 121, 123, 128, 137, 142, 143, 145, 152, 193, 274, 432]


## 运行结果与结论

上方输出显示：第 1 轮主问题为空集、子问题返回 200 个未覆盖行；加入 200 条逻辑割后，第 2 轮主问题解得目标 **429.0** 且覆盖全部行，收敛。总迭代 2 轮、200 条割。

**基准最优值来源**：直接 MIP（01_direct）证明最优值 429.0；LBBD 主问题最终与完整 SCP 等价，也证明最优 429.0。

## 结论

覆盖型问题的 LBBD 逻辑割 $\sum_{j:r\in S_j} y_j\ge 1$ 就是原覆盖约束本身，因此 LBBD 退化为带惰性约束的 MIP；它实现简单，2 轮收敛，是覆盖类问题很自然的分解形式。